# Lab 04 — Identity, capability, and authority

**PART II — Foundations of executable governance**  
*Ch. 5 — Identity, Capabilities, and Authority*

`intermediate` · about 20 minutes

## By the end of this lab you will be able to

- Explain why a framework role string cannot serve as a governance identity
- Resolve a framework name to a governance identity, and watch an unmapped one fail
- Distinguish capability, permission, and authority
- Distinguish delegation from handoff and predict the escalation defect

**Concepts:** `governance identity`, `framework binding`, `capability`, `ambient authority`, `least privilege`, `delegation`, `handoff`, `confused deputy`

---

Run the cell below. It executes the *same* `lab.py` the CLI runs — this notebook is a second view onto one implementation, not a copy, so the two can never disagree.


In [1]:
# Make the repository importable from anywhere under notebooks/
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from nornyx_lab.engine import find_lab, run_lab
meta = find_lab('04')
print(meta.title)

Identity, capability, and authority


## Run the lab


In [2]:
ctx = run_lab(find_lab('04'))

  Lab 04    Identity, capability, and authority

  Textbook: Ch. 5 — Identity, Capabilities, and Authority

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

▸ Why a role string is not an identity

  the gap between a framework name and a governance identity

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│     - id: delegation.refund_proposal                                                                            │
│       delegator_ref: identity.case_analyst                                                                      │
│       delegate_ref: identity.remediation_agent                                                                  │
│       capability_ref: propose_refund                                                                            │
│       purpose: Delegate bounded refund proposals to the remediation specialist.                                 │
│       actions: [propose_refund]                                                                                 │
│       scope_refs: [RemediationContext]                                                                          │
│       status: active                                                                                            │
│       valid_from: "2026-01-01T00:00:00Z"                                                                        │
│       expires_at: "2026-12-01T00:00:00Z"                                                                        │
│       max_depth: 1                                                                                              │
│       current_depth: 0                                                                                          │
│       onward_delegation: denied                                                                                 │
│       source_zone_ref: zone.remediation_internal                                                                │
│       target_zone_ref: zone.remediation_internal                                                                │
│       required_gate_refs: [gate.refund_review]                                                                  │
│       required_policy_refs: [RemediationGovernance]                                                             │
│       required_approval_refs: []                                                                                │
│       required_evidence_refs: [agentic_network_contract_re                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Four bounds at once — capability_ref, scope_refs, expires_at, max_depth: 1 — plus               
onward_delegation: denied. That last line is what stops a two-hop chain from becoming a six-hop 
one nobody modelled.

╭─────────────────────────────────────────── concept · confused deputy ───────────────────────────────────────────╮
│ A component with more authority than its caller, that acts on the caller's instructions without re-checking     │
│ whose authority applies. The classic agentic shape: a "tool executor" service holding broad credentials,        │
│ invoked by an agent that holds almost none.                                                                     │
│                                                                                                                 │
│ The design property that removes it: the executor must act under the caller's capability, not its own. In       │
│ contract terms — the decision is evaluated against identity.case_analyst, never against the process that        │
│ happens to run the tool.                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ⊘ where this stops ───────────────────────────────────────────────╮
│ resolve_identity maps a claimed framework name to a declared identity. It does not authenticate anything. If    │
│ your adapter passes agent_key="ceo_agent", Nornyx will resolve it and evaluate it as that identity.             │
│                                                                                                                 │
│ Binding a runtime actor to a claim is the platform's job — process identity, workload identity, mTLS. Nornyx    │
│ states the mapping; something else must prove the claim. Getting this backwards is the most consequential       │
│ misreading of the whole SPI.                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── ⚑ your turn ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  1 In contracts/ledger/network.nyx, add crewai:auditor as a framework binding on identity.compliance_officer.   │
│    Rebuild with python scripts/build_contracts.py ledger, re-run, and watch it resolve.                         │
│  2 Now try to give identity.intake_agent the issue_refund capability by editing capability_refs. Rebuild. What  │
│    does the membership block do to your change, and why is holding it in two places not redundant?              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Inspect what the lab measured

Every lab publishes its findings with `ctx.record(...)`. This is the same data `checks.py` asserts on — poke at it.


In [ ]:
import json
print(json.dumps(ctx.results, indent=2, default=str))

## Prove it

The concept checks for this lab. Each one is a proposition written so a machine can settle it.


In [ ]:
import subprocess, sys
checks = ROOT / 'labs' / '04_identity_and_capability' / 'checks.py'
proc = subprocess.run(
    [sys.executable, '-m', 'pytest', str(checks), '-v', '--no-header'],
    cwd=str(ROOT), capture_output=True, text=True,
    encoding='utf-8', errors='replace',
)
print(proc.stdout[-4000:])

## Your turn

The lab printed a **your turn** panel above. Do it here — edit the contract, re-run the cells, and watch which decision changes.

---

Next: [Lab 05 — Trust zones, and prompt injection as authority confusion](./05_trust_zones.ipynb)


In [ ]:
# scratch space
